In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


d:\AI\Assignment_1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

MODEL = "openai/gpt-oss-120b"


# Database Setup

In [3]:
import sqlite3
from datetime import datetime

In [4]:
DB_NAME = "finance_tracker.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS salary (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            amount REAL,
            month TEXT,
            year INTEGER,
            created_at TEXT)
        ''')
    
    c.execute('''
        CREATE TABLE IF NOT EXISTS expenses (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            amount REAL,
            category TEXT,
            description TEXT,
            date TEXT,
            created_at TEXT)
        ''')
    
    conn.commit()
    conn.close()

init_db()    

print("Database initialized successfully!")

Database initialized successfully!


# Implement the Tool Functions

## set_salary

In [5]:
def set_salary(amount, month, year):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    created_at = datetime.now().isoformat()
    c.execute('''
        INSERT INTO salary (amount, month, year, created_at)
        VALUES (?, ?, ?, ?)
    ''', (amount, month, year, created_at))
    conn.commit()
    conn.close()
    return f"Salary of ${amount:,.2f} for {month} {year} saved."

In [6]:
print(set_salary(60000, "April", 2026))

Salary of $60,000.00 for April 2026 saved.


## log_expense

In [7]:
def log_expense(amount, category, description, date):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    created_at = datetime.now().isoformat()
    c.execute('''
        INSERT INTO expenses (amount, category, description, date, created_at)
        VALUES (?, ?, ?, ?, ?)
    ''', (amount, category, description, date, created_at))

    conn.commit()

    # Calculate balance
    
    current_month = datetime.now().strftime("%B")
    current_year = datetime.now().year
    c.execute('SELECT SUM(amount) FROM salary WHERE month = ? AND year = ?', (current_month, current_year))
    salary_row = c.fetchone()
    salary = salary_row[0] if salary_row and salary_row[0] is not None else 0

    c.execute('SELECT SUM(amount) FROM expenses WHERE date LIKE ?', (f'{current_year}-%',))
    total_spent = c.fetchone()[0] or 0

    conn.close()

    remaining_balance = salary - total_spent
    return f"Logged expense of ${amount:,.2f} for {category} - {description} on {date}| Spent: ${total_spent:,.2f} | Remaining balance for {current_month} {current_year}: ${remaining_balance:,.2f}"

In [8]:
print(log_expense(60, "Transport", "Fuel", "2026-04-20"))

Logged expense of $60.00 for Transport - Fuel on 2026-04-20| Spent: $60,460.00 | Remaining balance for April 2026: $199,140.00


## get_balance

In [9]:
def get_balance():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    current_month = datetime.now().strftime("%B")
    current_year = datetime.now().year
    c.execute('SELECT SUM(amount) FROM salary WHERE month = ? AND year = ?', (current_month, current_year))
    salary_row = c.fetchone()
    salary = salary_row[0] if salary_row and salary_row[0] is not None else 0

    c.execute('SELECT SUM(amount) FROM expenses WHERE date LIKE ?', (f'{current_year}-%',))
    total_spent = c.fetchone()[0] or 0

    conn.close()

    remaining_balance = salary - total_spent

    return f"Salary for {current_month} {current_year}: ${salary:,.2f} | Total spent: ${total_spent:,.2f} | Remaining balance: ${remaining_balance:,.2f}"

In [10]:
print(get_balance())

Salary for April 2026: $259,600.00 | Total spent: $60,460.00 | Remaining balance: $199,140.00


## get_expense_summary

In [11]:
def get_expense_summary():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    current_month = datetime.now().strftime("%B")   
    current_year = datetime.now().year
    
    # Get salary for current month/year
    c.execute('SELECT SUM(amount) FROM salary WHERE month = ? AND year = ?', (current_month, current_year))
    salary_row = c.fetchone()
    salary = salary_row[0] if salary_row and salary_row[0] is not None else 0
    
    # Get total spent for current year
    c.execute('SELECT SUM(amount) FROM expenses WHERE date LIKE ?', (f'{current_year}-%',))
    total_spent = c.fetchone()[0] or 0
    
    c.execute('SELECT category, SUM(amount) FROM expenses WHERE date LIKE ? GROUP BY category', (f'{current_year}-%',))
    summary = c.fetchall()
    conn.close()
    

    summary_str = f"Expense Summary for {current_month} {current_year}:\n"
    for category, amount in summary:
        pct = (amount / salary) * 100 if salary > 0 else 0
        summary_str += f"- {category}: ${amount:,.2f} ({pct:.1f}% of salary)\n"
    summary_str += f"- Total Salary: ${salary:,.2f}\n- Total Spent: ${total_spent:,.2f}\n- Remaining Balance: ${(salary - total_spent):,.2f}"

    return summary_str

In [12]:
print(get_expense_summary())

Expense Summary for April 2026:
- Transport: $60,460.00 (23.3% of salary)
- Total Salary: $259,600.00
- Total Spent: $60,460.00
- Remaining Balance: $199,140.00


## Describe Your Tools to the AI

In [13]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "set_salary",
            "description": "Save or update the user's monthly salary/income. Call this when the user mentions their salary, income, earnings, or pay for a month.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Salary amount in dollars"},
                    "month": {"type": "string", "description": "Month name, e.g. April"},
                    "year": {"type": "integer", "description": "Year, e.g. 2026"}
                },
                "required": ["amount", "month", "year"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "log_expense",
            "description": "Log a new expense. Call this when the user mentions spending money, buying something, paying a bill, or any purchase.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Expense amount in dollars"},
                    "category": {"type": "string", "description": "Category like Groceries, Rent, Transport, Food, Entertainment"},
                    "description": {"type": "string", "description": "Short description of the expense"},
                    "date": {"type": "string", "description": "Date in YYYY-MM-DD format, use today if not mentioned"}
                },
                "required": ["amount", "category", "description"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get the current month's balance — salary, total spent, and remaining. Call this when the user asks how much is left, their remaining budget, or how they are tracking.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_expense_summary",
            "description": "Get a breakdown of expenses by category for the current month. Call this when the user asks for a summary, breakdown, or wants to see where they spent money.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    }
]


In [14]:
tools

[{'type': 'function',
  'function': {'name': 'set_salary',
   'description': "Save or update the user's monthly salary/income. Call this when the user mentions their salary, income, earnings, or pay for a month.",
   'parameters': {'type': 'object',
    'properties': {'amount': {'type': 'number',
      'description': 'Salary amount in dollars'},
     'month': {'type': 'string', 'description': 'Month name, e.g. April'},
     'year': {'type': 'integer', 'description': 'Year, e.g. 2026'}},
    'required': ['amount', 'month', 'year']}}},
 {'type': 'function',
  'function': {'name': 'log_expense',
   'description': 'Log a new expense. Call this when the user mentions spending money, buying something, paying a bill, or any purchase.',
   'parameters': {'type': 'object',
    'properties': {'amount': {'type': 'number',
      'description': 'Expense amount in dollars'},
     'category': {'type': 'string',
      'description': 'Category like Groceries, Rent, Transport, Food, Entertainment'},
   

## The Tool Calling Loop

In [15]:
groq = OpenAI(
    api_key=groq_api_key, 
    base_url="https://api.groq.com/openai/v1")  

MODEL = "openai/gpt-oss-120b"

TOOL_SET = {
    "set_salary": set_salary,
    "log_expenses": log_expense,    
    "get_balance": get_balance,
    "get_expense_summary": get_expense_summary
}

def chat(user_message, history):
    messages = [{"role": "system", "content": "You are a helpful personal finance assistant. "
    "You can help users track their salary and expenses, and provide summaries of their financial status."}]

    for msg, assistant in history:
        messages.append({"role": "user", "content": msg})
        messages.append({"role": "assistant", "content": assistant})

    messages.append({"role": "user", "content": user_message})

    
    while True:
        response = groq.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
    )

        message = response.choices[0].message

        if not message.tool_calls:
            return message.content
    
        messages.append(message)

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            
            tool_args = {k: v for k, v in tool_args.items()}

            result = TOOL_SET[tool_name](**tool_args)

            messages.append({
                "role": "tool", 
                "tool_call_id": tool_call.id, 
                "content": result
                })


## Build the Gradio UI

In [16]:
import gradio as gr

hisrory = []

with gr.Blocks(title="Personal_Finance Tracker") as demo:
    gr.Markdown("# Personal Finance Tracker \n Chat with your AI finance assistant")

    chatbot = gr.Chatbot()
    ms = gr.Textbox(placeholder="ex: My monthly salary is $3,500 I earn $4,000 a month Set my income to 2800")
    
    clear = gr.Button("Clear")
    submit = gr.Button("Submit")


    def respond(message, chat_history):
        global hisrory

        hisrory = []

        for item in chat_history:
            if isinstance(item, dict):
                if item.get('role') == 'user':
                    hisrory.append((item.get('content', ''), ''))
            elif item.get('role') == 'assistant' and hisrory:
                    hisrory[-1] = (hisrory[-1][0], item['content'])

        response = chat(message, hisrory)


        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": response})
                    
        return chat_history, ""
    
    ms.submit(respond, [ms, chatbot], [chatbot, ms])
    submit.click(respond, [ms, chatbot], [chatbot, ms])

    clear.click(lambda: ([], ""),None, [chatbot, ms])

demo.launch(inbrowser=True, auth=("stw", "12345"))


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
